# Inference Performance Table
Reads `*_perf.yaml` files from each PDE's performance directory,
computes derived metrics, and outputs the LaTeX table.

In [10]:
# ── Cell 1 — Imports ──────────────────────────────────────────────────────────
import yaml
import numpy as np
import pandas as pd
from pathlib import Path

In [11]:
# ── Cell 2 — Config: edit these paths / patterns ──────────────────────────────

PROJECT_ROOT = Path("/Users/gabrieleaccarino/Workspace/waveletflux")

PDES = [
    "turbulent_radiative_layer_2D",
    "rayleigh_benard",
    "active_matter",
]

# Glob prefixes to discover YAML files inside each performance/ directory.
# A file matches if its name starts with one of these strings.
MODEL_PATTERNS = [
    "FM_data_space",
    "WaveletFlow_1scale",
    "WaveletFlow_3scales",
]

# NFE used during inference (same for all models in this benchmark)
NFE = 50

In [12]:
# ── Cell 3 — YAML loading helpers ────────────────────────────────────────────

def avg_gpu_mem(block):
    """Return mean peak GPU memory across all ranks (GiB)."""
    if block is None:
        return None
    if isinstance(block, (int, float)):
        return float(block)
    if isinstance(block, dict):
        vals = [float(v) for v in block.values() if v is not None]
        return float(np.mean(vals)) if vals else None
    return None


def resolve_model_name(filename: str, pde: str) -> str:
    """Strip the known _{pde}_perf.yaml suffix to recover the model name."""
    for suffix in (f"_{pde}_perf.yaml", f"_{pde}_performance.yaml"):
        if filename.endswith(suffix):
            return filename[: -len(suffix)]
    return Path(filename).stem


def load_yaml(path: Path) -> dict:
    with open(path) as f:
        raw = yaml.safe_load(f)
    wc = raw.get("wall_clock_s")
    ts = raw.get("total_steps_global")
    return {
        "wall_clock_s":        wc,
        "total_steps_global":  ts,
        "gpu_peak_mem_GiB":    avg_gpu_mem(raw.get("gpu_peak_mem_GiB")),
        "throughput_steps_s":  raw.get("throughput_steps_s"),
        "throughput_frames_s": raw.get("throughput_frames_s"),
        # derived
        "sec_per_rollout":     round(wc / ts, 6) if (wc and ts) else None,
    }

In [13]:
# ── Cell 4 — Discover and load all YAML files ─────────────────────────────────

records = []

for pde in PDES:
    perf_dir = PROJECT_ROOT / "experiments" / pde / "performance"
    if not perf_dir.exists():
        print(f"[MISS] {perf_dir}")
        continue

    found: dict[str, Path] = {}
    for pattern in MODEL_PATTERNS:
        for p in sorted(perf_dir.glob(f"{pattern}*_perf.yaml")) + \
                 sorted(perf_dir.glob(f"{pattern}*_performance.yaml")):
            name = resolve_model_name(p.name, pde)
            found.setdefault(name, p)      # first match wins

    if not found:
        print(f"[MISS] no matching YAMLs in {perf_dir}")
        continue

    for model_name, yaml_path in sorted(found.items()):
        print(f"  ✓  {yaml_path.name}")
        try:
            m = load_yaml(yaml_path)
            records.append({"pde": pde, "model": model_name, **m})
        except Exception as e:
            print(f"    [ERROR] {yaml_path}: {e}")

df = pd.DataFrame(records)
df["throughput_Mframes_s"] = df["throughput_frames_s"] / 1e6
print(f"\nLoaded {len(df)} rows")
df

  ✓  FM_data_space_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  FM_data_space_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_1scale_db4_v4.0_nsampling50_nen

,pde,model,wall_clock_s,total_steps_global,gpu_peak_mem_GiB,throughput_steps_s,throughput_frames_s,sec_per_rollout,throughput_Mframes_s
0,turbulent_radiative_layer_2D,FM_data_space_v4.0_nsampling50_nens8,247.75,6984,0.59000,28.2,1385580.0,0.035474,1.385580
1,turbulent_radiative_layer_2D,WaveletFlow_1scale_db2_v4.0_nsampling50_nens8,199.04,6984,0.59200,35.1,1724697.7,0.028499,1.724698
2,turbulent_radiative_layer_2D,WaveletFlow_1scale_db4_v4.0_nsampling50_nens8,208.10,6984,0.59100,33.6,1649548.1,0.029797,1.649548
3,turbulent_radiative_layer_2D,WaveletFlow_1scale_db6_v4.0_nsampling50_nens8,204.52,6984,0.59200,34.1,1678469.7,0.029284,1.678470
4,turbulent_radiative_layer_2D,WaveletFlow_1scale_haar_v4.0_nsampling50_nens8,198.06,6984,0.59100,35.3,1733232.8,0.028359,1.733233
5,turbulent_radiative_layer_2D,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,224.02,6984,0.66300,31.2,1532380.0,0.032076,1.532380
6,turbulent_radiative_layer_2D,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,228.23,6984,0.66300,30.6,1504072.0,0.032679,1.504072
7,turbulent_radiative_layer_2D,WaveletFlow_3scales_db6_v4.0_nsampling50_nens8,221.05,6984,0.66300,31.6,1552952.8,0.031651,1.552953
8,turbulent_radiative_layer_2D,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,218.27,6984,0.66300,32.0,1572733.1,0.031253,1.572733
9,rayleigh_benard,FM_data_space_v4.0_nsampling50_nens8,633.85,14112,1.44100,22.3,1459093.4,0.044916,1.459093


In [ ]:
# ── Cell 5 — Classify models and compute WFM-vs-FM ratios ────────────────────

WFM_KEYWORDS = ["WaveletFlow", "WFM", "Wavelet"]

def is_wfm(name: str) -> bool:
    return any(k.lower() in name.lower() for k in WFM_KEYWORDS)

df["model_type"] = df["model"].apply(lambda m: "WFM" if is_wfm(m) else "FM")

# Ratio metrics: direction="lower" → FM/WFM  |  "higher" → WFM/FM
RATIO_METRICS = [
    ("wall_clock_s",        "lower"),
    ("sec_per_rollout",     "lower"),
    ("gpu_peak_mem_GiB",    "lower"),
    ("throughput_Mframes_s","higher"),
]

for m, _ in RATIO_METRICS:
    df[f"ratio_{m}"] = np.nan

# Replace df.groupby("pde") with df_best.groupby("pde") in the ratio loop
for pde, grp in df_best.groupby("pde"):
    fm = grp[grp["model_type"] == "FM"]
    if fm.empty:
        print(f"[WARN] no FM baseline for {pde}")
        continue
    fm_row = fm.iloc[0]
    for idx in grp[grp["model_type"] == "WFM"].index:
        for metric, direction in RATIO_METRICS:
            fv = fm_row.get(metric)
            wv = df_best.at[idx, metric]
            if pd.notna(fv) and pd.notna(wv) and fv != 0:
                df_best.at[idx, f"ratio_{metric}"] = (
                    wv / fv if direction == "higher" else fv / wv
                )

df[["pde","model","model_type",
    "wall_clock_s","sec_per_rollout","gpu_peak_mem_GiB","throughput_Mframes_s",
    "ratio_wall_clock_s","ratio_gpu_peak_mem_GiB"]]

# ── Cell 5b — Define best WFM config per PDE ─────────────────────────────────
PDE_BEST_CONFIG = {
    "turbulent_radiative_layer_2D": {
        "WaveletFlow_1scale":  "WaveletFlow_1scale_haar",
        "WaveletFlow_3scales": "WaveletFlow_3scales_haar",
    },
    "rayleigh_benard": {
        "WaveletFlow_1scale":  "WaveletFlow_1scale_db4",
        "WaveletFlow_3scales": "WaveletFlow_3scales_db4",
    },
    "active_matter": {
        "WaveletFlow_1scale":  "WaveletFlow_1scale_haar",
        "WaveletFlow_3scales": "WaveletFlow_3scales_db2",
    },
}

def is_best_wfm(row) -> bool:
    """Keep a WFM row only if it matches the best config for its PDE."""
    if not is_wfm(row["model"]):
        return True   # always keep FM baseline
    pde_cfg = PDE_BEST_CONFIG.get(row["pde"], {})
    for scale_key, best_prefix in pde_cfg.items():
        if scale_key in row["model"] and row["model"].startswith(best_prefix):
            return True
    return False

df_best = df[df.apply(is_best_wfm, axis=1)].copy()

In [15]:
# ── Cell 6 — Label maps and display pivot ─────────────────────────────────────

MODEL_LABELS = {
    "FM_data_space":      r"$\mathrm{FM}_{\mathrm{pixel}}$",
    "WaveletFlow_1scale": "WFM 1-scale",
    "WaveletFlow_3scales":"WFM 3-scale",
}

def shorten_model(name: str) -> str:
    """Map e.g. WaveletFlow_1scale_haar_v4.0 → WFM 1-scale."""
    for key, label in MODEL_LABELS.items():
        if name.startswith(key):
            return label
    return name


MODEL_ORDER = list(MODEL_LABELS.values())
PDE_LABELS  = {
    "turbulent_radiative_layer_2D": "Turb. Rad. Layer 2D",
    "rayleigh_benard":              "Rayleigh-Bénard",
    "active_matter":                "Active Matter",
}
PDE_ORDER = list(PDE_LABELS.values())

df2 = df_best.copy()   # ← was df.copy()
df2["model_short"] = df2["model"].apply(shorten_model)
df2["pde_label"]   = df2["pde"].map(PDE_LABELS)

# ── Add wavelet annotation column for caption reference ───────────────────────
def extract_wavelet(name: str) -> str:
    for w in ["haar", "db2", "db4", "db6"]:
        if w in name:
            return w
    return ""

df2["wavelet"] = df2["model"].apply(extract_wavelet)


# (df_column, display_name, format_spec, show_for_FM)
METRICS = [
    ("wall_clock_s",        "Wall Clock (s)",     ".1f",  True),
    ("sec_per_rollout",     "sec / step",         ".4f",  True),
    ("gpu_peak_mem_GiB",    "GPU Mem (GiB)",      ".2f",  True),
    ("throughput_Mframes_s","Throughput (Mfr/s)", ".2f",  True),
    ("ratio_wall_clock_s",  "Speedup ×",          ".2f",  False),
    ("ratio_gpu_peak_mem_GiB","Mem ratio ×",      ".2f",  False),
]
METRIC_COLS = [label for _, label, _, _ in METRICS]
fmt_map     = {label: fmt  for _, label, fmt, _    in METRICS}

rows, index = [], []
for pde_label in PDE_ORDER:
    for model in MODEL_ORDER:
        is_fm = "FM" in model and "WFM" not in model
        cell  = df2[(df2["pde_label"] == pde_label) & (df2["model_short"] == model)]
        row   = {}
        for col, label, fmt, show_for_fm in METRICS:
            if cell.empty or col not in cell.columns:
                row[label] = np.nan
            elif is_fm and not show_for_fm:
                row[label] = np.nan
            else:
                row[label] = cell[col].values[0]
        rows.append(row)
        index.append((pde_label, model))

pivot = pd.DataFrame(rows, columns=METRIC_COLS,
                     index=pd.MultiIndex.from_tuples(index, names=["PDE", "Model"]))

def fmt_cell(val, metric_name):
    if pd.isna(val):
        return "—"
    result = f"{val:{fmt_map.get(metric_name, '.3f')}}"
    if "×" in metric_name:
        result += "×"
    return result

pivot_fmt = pivot.copy().astype(object)
for metric in METRIC_COLS:
    pivot_fmt[metric] = pivot[metric].apply(lambda v: fmt_cell(v, metric))

pivot_fmt

Wall Clock (s) sec / step  \
PDE                 Model                                                      
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$          247.8     0.0355   
                    WFM 1-scale                             198.1     0.0284   
                    WFM 3-scale                             218.3     0.0313   
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$          633.9     0.0449   
                    WFM 1-scale                             418.1     0.0296   
                    WFM 3-scale                             457.9     0.0324   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$          310.8     0.0459   
                    WFM 1-scale                             213.3     0.0315   
                    WFM 3-scale                             247.4     0.0365   

                                                   GPU Mem (GiB)  \
PDE                 Model                                          
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$          0.59   
                    WFM 1-scale                             0.59   
                    WFM 3-scale                             0.66   
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$          1.44   
                    WFM 1-scale                             1.44   
                    WFM 3-scale                             1.51   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$          1.63   
                    WFM 1-scale                             1.64   
                    WFM 3-scale                             1.71   

                                                   Throughput (Mfr/s)  \
PDE                 Model                                               
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$               1.39   
                    WFM 1-scale                                  1.73   
                    WFM 3-scale                                  1.57   
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$               1.46   
                    WFM 1-scale                                  2.21   
                    WFM 3-scale                                  2.02   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$               1.43   
                    WFM 1-scale                                  2.08   
                    WFM 3-scale                                  1.80   

                                                   Speedup × Mem ratio ×  
PDE                 Model                                                 
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 1-scale                        1.25×       1.00×  
                    WFM 3-scale                        1.14×       0.89×  
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 1-scale                        1.52×       1.00×  
                    WFM 3-scale                        1.38×       0.95×  
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 1-scale                        1.46×       1.00×  
                    WFM 3-scale                        1.26×       0.95×

In [7]:
# ── Cell 7 — LaTeX table ──────────────────────────────────────────────────────

PDE_LATEX = {
    "Turb. Rad. Layer 2D": "Turbulence Radiative Layer 2D",
    "Rayleigh-Bénard":     "Rayleigh-B\\'{e}nard",
    "Active Matter":       "Active Matter",
}

MODEL_LATEX = {
    r"$\mathrm{FM}_{\mathrm{pixel}}$": r"$FM_{pixel}$",
    "WFM 1-scale":  "WFM 1-scale",
    "WFM 3-scales": "WFM 3-scales",
}

caption = (
    r"Inference performance across PDEs and models. "
    r"\textit{Wall Clock}: total inference time (max across GPU ranks). "
    r"\textit{sec/step}: wall-clock time per autoregressive rollout step "
    r"(\texttt{wall\_clock\_s} $/ $ \texttt{total\_steps\_global}), "
    r"resolution-independent. "
    r"\textit{GPU Mem}: peak GPU memory averaged across ranks. "
    r"\textit{Throughput}: millions of spatial frames per second "
    r"(\texttt{total\_steps} $\times H \times W$ / \texttt{wall\_clock\_s}). "
    r"\textit{Speedup}: wall-clock ratio FM\,/\,WFM; ${>}1$ means WFM is faster. "
    r"\textit{Mem ratio}: peak GPU memory ratio FM\,/\,WFM; ${>}1$ means WFM uses less memory. "
    r"All runs use NFE\,=\,50 ODE steps, 4 GPUs, and 8 ensemble members."
)

lines = []
lines.append(r"\begin{table}[ht]")
lines.append(r"\caption{" + caption + "}")
lines.append(r"\label{tab:inference_perf}")
lines.append(r"\resizebox{\textwidth}{!}{%")
lines.append(r"\begin{tabular}{ll rrrr rr}")
lines.append(r"\toprule")
lines.append(
    r"\textbf{PDE} & \textbf{Model} "
    r"& \textbf{Wall Clock (s)} "
    r"& \textbf{sec / step} "
    r"& \textbf{GPU Mem (GiB)} "
    r"& \textbf{Throughput (Mfr/s)} "
    r"& \textbf{Speedup $\times$} "
    r"& \textbf{Mem ratio $\times$} \\\\"
)

for pde_label in PDE_ORDER:
    lines.append(r"\midrule")
    pde_latex = PDE_LATEX.get(pde_label, pde_label)
    models_in_pde = MODEL_ORDER
    lines.append(rf"\multirow{{{len(models_in_pde)}}}{{*}}{{{pde_latex}}}")

    for i, model in enumerate(models_in_pde):
        model_latex = MODEL_LATEX.get(model, model)
        row_data    = pivot_fmt.loc[(pde_label, model)]
        vals        = " & ".join(
            str(row_data[m]).replace("×", r"$\times$") for m in METRIC_COLS
        )
        # replace — with proper LaTeX dash
        vals = vals.replace("—", r"\textemdash")
        pde_col = "" if i > 0 else ""   # multirow handles the PDE cell
        lines.append(f"  & {model_latex} & {vals} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}%")
lines.append("}")
lines.append(r"\end{table}")

latex = "\n".join(lines)
print(latex)

\begin{table}[ht]
\caption{Inference performance across PDEs and models. \textit{Wall Clock}: total inference time (max across GPU ranks). \textit{sec/step}: wall-clock time per autoregressive rollout step (\texttt{wall\_clock\_s} $/ $ \texttt{total\_steps\_global}), resolution-independent. \textit{GPU Mem}: peak GPU memory averaged across ranks. \textit{Throughput}: millions of spatial frames per second (\texttt{total\_steps} $\times H \times W$ / \texttt{wall\_clock\_s}). \textit{Speedup}: wall-clock ratio FM\,/\,WFM; ${>}1$ means WFM is faster. \textit{Mem ratio}: peak GPU memory ratio FM\,/\,WFM; ${>}1$ means WFM uses less memory. All runs use NFE\,=\,50 ODE steps, 4 GPUs, and 8 ensemble members.}
\label{tab:inference_perf}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll rrrr rr}
\toprule
\textbf{PDE} & \textbf{Model} & \textbf{Wall Clock (s)} & \textbf{sec / step} & \textbf{GPU Mem (GiB)} & \textbf{Throughput (Mfr/s)} & \textbf{Speedup $\times$} & \textbf{Mem ratio $\times$} \\\\
\

In [8]:
# ── Cell 8 — Save to .tex file ────────────────────────────────────────────────
#out = Path("performance_table.tex")
#out.write_text(latex)
#print(f"Saved → {out.resolve()}")